# Handwriting with a 2-joint muscle-driven arm (MotorNet)

Same logic as the centre-out reach notebook, but:

- **Plant**: the simple built-in 2-DoF arm (`TwoDofArm`) = shoulder + elbow, driven by 6 redundant
  muscles.
- **Task**: a custom `Handwriting` environment (in `MotorNetUtils/environment.py`) where the target
  is a point that travels, at constant speed, along the stroke path that spells a **word**. Tracking
  that moving target makes the hand *write the word*.
- **Controller**: a GRU policy trained end-to-end through the differentiable simulation.

The word is drawn with a tiny single-stroke vector font and centred on the hand's start position.

In [ ]:
import sys, pathlib
# make the repo root importable whether the kernel runs from the repo root or from notebooks/
_root = pathlib.Path.cwd()
_root = _root if (_root / "MotorNetUtils").exists() else _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import torch
import matplotlib.pyplot as plt
import motornet as mn
from MotorNetUtils.environment import Handwriting

device = torch.device("cpu")
torch.manual_seed(0)
np.random.seed(0)
print("motornet", mn.__version__)

## 1. Build the plant: the simple 2-DoF arm

MotorNet's built-in `TwoDofArm` (shoulder + elbow) with 6 redundant muscles attached via
`add_muscle`. Each muscle is `(fixation_bodies, coordinates, max_force, name)`; bodies are
`0` = world, `1` = upper arm, `2` = forearm.

In [ ]:
effector = mn.effector.Effector(
    skeleton=mn.skeleton.TwoDofArm(),
    muscle=mn.muscle.ReluMuscle(),
    timestep=0.01,
)

muscles = [
    ([0, 1], [[0.0,  0.04], [0.12, 0.0]], 1000, "sho_flx"),
    ([0, 1], [[0.0, -0.04], [0.12, 0.0]], 1000, "sho_ext"),
    ([1, 2], [[0.28,  0.03], [0.10, 0.0]], 600, "elb_flx"),
    ([1, 2], [[0.28, -0.03], [0.10, 0.0]], 600, "elb_ext"),
    ([0, 2], [[0.0,  0.03], [0.10, 0.0]],  500, "bi_flx"),
    ([0, 2], [[0.0, -0.03], [0.10, 0.0]],  500, "bi_ext"),
]
for fixation, coords, force, name in muscles:
    effector.add_muscle(path_fixation_body=fixation, path_coordinates=coords,
                        max_isometric_force=force, name=name)

print("degrees of freedom:", effector.skeleton.dof)
print("number of muscles :", effector.n_muscles)

## 2. The handwriting task

`Handwriting` builds, for each episode, a target trajectory that traces the stroke path of a word.
The observation gives the controller the *current* point to be at (plus visual / proprioceptive
feedback). Change `words` to write something else — supported letters are an A-Z subset of a
single-stroke font (see `FONT` in `environment.py`).

In [ ]:
env = Handwriting(effector, words="hi", size=0.07,
                  vision=True, proprioception=True)

obs_dim = env.observation_space.shape[0]
n_muscles = env.n_muscles
print("observation dim:", obs_dim, "| action dim (muscles):", n_muscles)

# preview the target trajectory the hand is asked to trace
_, info = env.reset(options={"batch_size": 1})
ref = env.reference[0].numpy()
plt.figure(figsize=(6, 4))
plt.plot(ref[:, 0], ref[:, 1], "-", color="0.6", lw=4, solid_capstyle="round")
plt.plot(ref[0, 0], ref[0, 1], "o", color="g", ms=8, label="start")
plt.plot(ref[-1, 0], ref[-1, 1], "s", color="r", ms=8, label="end")
plt.gca().set_aspect("equal")
plt.title(f"target trajectory: {env.words_this_episode[0]!r}")
plt.xlabel("x (m)"); plt.ylabel("y (m)"); plt.legend()
plt.tight_layout(); plt.show()

## 3. Controller: a GRU policy

Same `PolicyGRU` as the reach notebook (sigmoid output, bias initialised to -5 for low initial
force). At each step it sees the current target point + feedback and outputs muscle activations.

In [ ]:
policy = mn.policy.PolicyGRU(obs_dim, 128, n_muscles, device=device)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)


def run_episode(batch_size):
    """Roll out one batch; return hand-xy, target-xy and actions over time."""
    h = policy.init_hidden(batch_size=batch_size)
    obs, info = env.reset(options={"batch_size": batch_size})
    xy, tg, acts = [], [], []
    terminated = False
    while not terminated:
        action, h = policy(obs, h)
        obs, _, terminated, _, info = env.step(action=action)
        xy.append(info["states"]["fingertip"][:, None, :])    # hand position
        tg.append(info["goal"][:, None, :])                   # current target point
        acts.append(action[:, None, :])
    return torch.cat(xy, dim=1), torch.cat(tg, dim=1), torch.cat(acts, dim=1)

## 4. Train

Loss = mean distance of the hand to the moving target over the whole trajectory, plus a small
effort penalty. Gradient norm is clipped to 1, and we skip the step if any gradient is non-finite
(a safe guard against singular postures).

In [ ]:
batch_size = 32
n_batches = 2000
losses = []

for b in range(n_batches):
    xy, tg, acts = run_episode(batch_size)
    pos_loss = torch.mean(torch.sum(torch.abs(xy - tg), dim=-1))
    effort = 1e-3 * torch.mean(torch.sum(acts ** 2, dim=-1))
    loss = pos_loss + effort

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
    if all(torch.isfinite(p.grad).all() for p in policy.parameters() if p.grad is not None):
        optimizer.step()

    losses.append(loss.item())
    if (b + 1) % 50 == 0:
        print(f"batch {b + 1:4d} | loss {loss.item():.4f} (pos {pos_loss.item():.4f})")

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("batch"); plt.ylabel("loss"); plt.title("training")
plt.tight_layout(); plt.show()

## 5. Read what the hand wrote

Run one episode and overlay the hand path (blue) on the target word (grey).

In [ ]:
with torch.no_grad():
    xy, tg, acts = run_episode(batch_size=1)
hand = xy[0].numpy()
word = env.reference[0].numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(word[:, 0], word[:, 1], "-", color="0.8", lw=6, solid_capstyle="round", label="target word")
ax.plot(hand[:, 0], hand[:, 1], "-", color="tab:blue", lw=2, label="hand path")
ax.plot(hand[0, 0], hand[0, 1], "o", color="k", ms=6, label="start")
ax.set_aspect("equal")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title(f"the hand writing {env.words_this_episode[0]!r}")
ax.legend()
plt.tight_layout(); plt.show()

### Try another word

Pass any word made of supported letters via the reset option `word` (no retraining needed if the
new word lives in a similar region; otherwise train with `words=` set to your target word/list).

In [ ]:
def write(word):
    h = policy.init_hidden(1)
    obs, info = env.reset(options={"batch_size": 1, "word": word})
    hand, terminated = [], False
    with torch.no_grad():
        while not terminated:
            a, h = policy(obs, h)
            obs, _, terminated, _, info = env.step(action=a)
            hand.append(info["states"]["fingertip"][0])
    hand = torch.stack(hand).numpy()
    ref = env.reference[0].numpy()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ref[:, 0], ref[:, 1], "-", color="0.8", lw=6, solid_capstyle="round")
    ax.plot(hand[:, 0], hand[:, 1], "-", color="tab:blue", lw=2)
    ax.set_aspect("equal"); ax.set_title(f"writing {word!r}")
    plt.tight_layout(); plt.show()


write("go")